# Reproducibility Notebook: Energy-Aware Sensing with RL

**Paper Revision for "Results in Engineering" Journal**

## Key Fix: Persistence Logic
- When sensor is **ON**: flag updates from ground truth
- When sensor is **OFF**: flag **persists** (stale value)

## Pareto Frontier Configurations
| Config | β Value | Target |
|--------|---------|--------|
| Safety-First | 0.05 | High Detection (~83%) |
| Balanced | 0.5 | Medium (~67% det, 68% save) |
| Energy-Saver | 1.0 | Max Savings (~87%) |

---
## 1. Setup

In [ ]:
!pip install -q numpy matplotlib

In [ ]:
import os, sys, pickle, random
import numpy as np
import matplotlib.pyplot as plt

if not os.path.exists('energy-aware-sensing-rl'):
    !git clone https://github.com/oussamaElallam/energy-aware-sensing-rl.git
os.chdir('energy-aware-sensing-rl')
sys.path.insert(0, '.')
from framework.rl_env import HealthWearableEnv

SENSOR_COSTS = [10, 4, 1]
print("✓ Setup complete!")

---
## 2. Load/Train Q-Tables

In [ ]:
beta_configs = {'Safety (0.05)': 0.05, 'Balanced (0.5)': 0.5, 'Saver (1.0)': 1.0}

Q_tables = {}
for name, beta in beta_configs.items():
    pkl_path = f'q_table_beta_{beta}.pkl'
    if os.path.exists(pkl_path):
        print(f"Loading {name}")
        with open(pkl_path, 'rb') as f:
            Q_tables[name] = pickle.load(f)
    else:
        print(f"Training {name} (not found)...")
        # Training code here if needed

print(f"\n✓ Loaded {len(Q_tables)} models")

---
## 3. Evaluate All Models

In [ ]:
def evaluate_policy(data, policy_fn):
    env = HealthWearableEnv(data=data, sensor_costs=SENSOR_COSTS, max_time_steps=len(data))
    state = env.reset()
    det_hits = det_total = energy = 0
    while not env.done:
        action = policy_fn(state)
        next_state, _, done, _ = env.step(action)
        ecg_on, ppg_on, tmp_on = (action >> 2) & 1, (action >> 1) & 1, action & 1
        energy += SENSOR_COSTS[0]*ecg_on + SENSOR_COSTS[1]*ppg_on + SENSOR_COSTS[2]*tmp_on
        if env.t <= len(data):
            gt = data[env.t - 1]
            for flag, on in [('arr_flag', ecg_on), ('bp_flag', ppg_on), ('fever_flag', tmp_on)]:
                if gt[flag]:
                    det_total += 1
                    if on: det_hits += 1
        if done: break
        state = next_state
    return (det_hits/det_total*100 if det_total > 0 else 0), energy * 5 / 3600

def greedy_policy(Q, state):
    return int(np.argmax([Q.get((state, a), 0.0) for a in range(8)]))

In [ ]:
results = {}
for name, Q in Q_tables.items():
    det_rates, energies = [], []
    for seed in range(10):
        rng = np.random.default_rng(seed)
        data = [{'arr_flag': int(rng.random() < 0.10),
                 'bp_flag': int(rng.random() < 0.30),
                 'fever_flag': int(rng.random() < 0.10)} for _ in range(12000)]
        det, energy = evaluate_policy(data, lambda s, Q=Q: greedy_policy(Q, s))
        det_rates.append(det)
        energies.append(energy)
    results[name] = {'det': np.mean(det_rates), 'det_std': np.std(det_rates),
                     'energy': np.mean(energies), 'energy_std': np.std(energies)}

results['Always-On'] = {'det': 100.0, 'det_std': 0, 'energy': 250.0, 'energy_std': 0}

print("\n" + "="*55)
print(f"{'Policy':<18} {'Detection':<14} {'Energy (mAh)':<14} {'Savings'}")
print("-"*55)
for name in ['Always-On', 'Safety (0.05)', 'Balanced (0.5)', 'Saver (1.0)']:
    r = results[name]
    sav = (1 - r['energy']/250)*100
    print(f"{name:<18} {r['det']:>5.1f}% ± {r['det_std']:>4.1f}  {r['energy']:>6.1f} ± {r['energy_std']:>4.1f}  {sav:>5.1f}%")
print("="*55)

---
## 4. Pareto Frontier Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

colors = {'Always-On': 'red', 'Safety (0.05)': 'green', 'Balanced (0.5)': 'blue', 'Saver (1.0)': 'purple'}
markers = {'Always-On': 's', 'Safety (0.05)': 'o', 'Balanced (0.5)': 'D', 'Saver (1.0)': '^'}

for name in ['Always-On', 'Safety (0.05)', 'Balanced (0.5)', 'Saver (1.0)']:
    r = results[name]
    ax.errorbar(r['energy'], r['det'], xerr=r['energy_std'], yerr=r['det_std'],
                fmt=markers[name], markersize=15, color=colors[name],
                label=name, capsize=5, capthick=2, elinewidth=2)

# Draw Pareto frontier line
pareto = [(results[n]['energy'], results[n]['det']) for n in ['Saver (1.0)', 'Balanced (0.5)', 'Safety (0.05)', 'Always-On']]
xs, ys = zip(*pareto)
ax.plot(xs, ys, 'k--', alpha=0.5, lw=2, label='Pareto Frontier')

ax.set_xlabel('Energy Consumption (mAh)', fontsize=14)
ax.set_ylabel('Detection Rate (%)', fontsize=14)
ax.set_title('Detection vs Energy Trade-off (Pareto Frontier)', fontsize=16)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 280)
ax.set_ylim(0, 110)

plt.tight_layout()
plt.savefig('pareto_frontier.png', dpi=150)
plt.show()
print("\n✓ Pareto plot saved!")

---
## Summary

| Config | β | Detection | Energy Savings |
|--------|---|-----------|----------------|
| Safety-First | 0.05 | 83% | 19% |
| Balanced | 0.5 | 67% | 68% |
| Energy-Saver | 1.0 | 33% | 87% |

**Conclusion**: The RL policy is **tunable** - adjusting β trades detection for energy savings.